# RV3 official xView2 rescore of joint_damage_full (CPU, zero GPU quota)

In [ ]:
import subprocess, sys, os, json, time, traceback
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = Path("/tmp/RescueVision")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/AliOmarAbdelhady/RescueVision.git", str(REPO)], check=True)
sys.path.insert(0, str(REPO / "src"))

def mark(stage, ok=True, extra=""):
    """Progress ledger in /kaggle/working: survives kills where the log does not."""
    rec = {"t": time.strftime("%H:%M:%S"), "stage": stage, "ok": bool(ok), "extra": str(extra)[:1500]}
    with open(WORK / "_progress.txt", "a") as f:
        f.write(json.dumps(rec) + "\n")
    (WORK / "_status.json").write_text(json.dumps(rec, indent=1))
    print("[mark]", rec, flush=True)

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
mark("setup:start")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "albumentations==2.0.8"], check=False)
import albumentations as _A
mark("setup:albumentations", extra=_A.__version__)
print("albumentations", _A.__version__)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch"], check=False)
import segmentation_models_pytorch as smp
print("smp", smp.__version__)
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NW = 0
PIN = False
print("torch", torch.__version__, "| device:", DEVICE)
print(subprocess.run(["df", "-h", "/tmp", "/kaggle/working"], capture_output=True, text=True).stdout)
mark("setup:done", extra=str(DEVICE))

import rescuevision.training.logging_utils as _lu
_orig_log = _lu.CSVLogger.log
def _marked_log(self, metrics, epoch):
    try:
        mark(f"epoch{epoch}", extra=str({k: round(float(v), 4) for k, v in metrics.items()
                                         if isinstance(v, (int, float))})[:400])
    except Exception:
        pass
    return _orig_log(self, metrics, epoch)
_lu.CSVLogger.log = _marked_log
print("epoch heartbeat patch applied")

In [ ]:
import numpy as np
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2

from rescuevision.constants import IMAGENET_MEAN, IMAGENET_STD
from rescuevision.data import datasets as _D
from PIL import Image as _PIL
from pathlib import Path as _PPath

def _load_arr(path):
    path = _PPath(path)
    if path.suffix == ".npy":
        return np.load(path)
    return np.array(_PIL.open(path))

def _find_tile(directory, tile_id):
    for ext in (".png", ".jpg", ".jpeg", ".npy"):
        p = _PPath(directory) / f"{tile_id}{ext}"
        if p.exists():
            return p
    raise FileNotFoundError(f"{tile_id} in {directory}")

def _tile_dataset_init(self, processed_dir, split_csv, split="train", mask_type="building_binary",
                       transform=None, image_size=512, debug_limit=None):
    import pandas as pd
    self.processed_dir = _PPath(processed_dir)
    self.images_dir = self.processed_dir / "images" / "pre"
    self.pre_dir = self.processed_dir / "images" / "pre"
    self.post_dir = self.processed_dir / "images" / "post"
    if type(self).__name__ == "XBDDamageSegmentationDataset":
        self.masks_dir = self.processed_dir / "masks" / "damage_multiclass"
    elif type(self).__name__ == "XBDChangeDetectionDataset":
        self.masks_dir = self.processed_dir / "masks" / "change_binary"
    else:
        self.masks_dir = self.processed_dir / "masks" / mask_type
    self.transform = transform
    self.image_size = image_size
    splits_df = pd.read_csv(split_csv)
    tile_ids = splits_df[splits_df["split"] == split]["tile_id"].tolist()
    if debug_limit:
        tile_ids = tile_ids[:debug_limit]
    self.tile_ids = [t for t in tile_ids
                     if _find_tile(self.pre_dir, t).exists()
                     and _find_tile(self.post_dir, t).exists()
                     and _find_tile(self.masks_dir, t).exists()]
    print(f"{type(self).__name__}: split={split}, {len(self.tile_ids)} tiles")

def _xbd_seg_getitem(self, idx):
    tile_id = self.tile_ids[idx]
    image = _load_arr(_find_tile(self.pre_dir, tile_id))
    mask = _load_arr(_find_tile(self.masks_dir, tile_id))
    if self.transform:
        aug = self.transform(image=image, mask=mask)
        image, mask = aug["image"], aug["mask"]
    mask = mask.long() if torch.is_tensor(mask) else torch.from_numpy(mask).long()
    return {"image": image, "mask": mask, "tile_id": tile_id}

_D.XBDSegmentationDataset.__init__ = _tile_dataset_init
_D.XBDSegmentationDataset.__getitem__ = _xbd_seg_getitem
mark("patch:done")
print("png-aware dataset patch applied")

In [ ]:
# ---- official xView2 pixel scorer (inlined verbatim from repo = single source) ----
"""Official xView2 pixel-based scoring.

Port of the metric used by the xView2 challenge (DIUx-xView/xView2_scoring):
identical semantics, array-based API so it can score in-memory predictions
without writing the official PNG directory layout.

Scoring rules (match the official implementation exactly):
- Localization F1: pixel counts of binarized building masks
  (pred building > 0 vs target building > 0), summed over ALL tiles globally.
- Damage F1: per tile, predicted damage is zeroed outside PREDICTED buildings
  (gating), then only pixels inside TARGET buildings (damage target > 0) are
  scored; per class 1-4 TP/FN/FP are summed globally and converted to F1.
- damage_f1 = harmonic mean of the four class F1s (epsilon 1e-6).
- score = 0.3 * localization_f1 + 0.7 * damage_f1.
"""

from __future__ import annotations

import numpy as np

__all__ = ["XView2PixelScorer", "f1_from_counts", "harmonic_mean_f1"]

_EPS = 1e-6


def f1_from_counts(tp: int, fp: int, fn: int) -> float:
    """F1 from global TP/FP/FN counts (0 when TP, precision or recall is 0)."""
    if tp <= 0:
        return 0.0
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    if precision == 0.0 or recall == 0.0:
        return 0.0
    return float(2 * precision * recall / (precision + recall))


def harmonic_mean_f1(f1s) -> float:
    """Official damage F1: harmonic mean of the four class F1s (eps 1e-6)."""
    xs = list(f1s)
    return float(len(xs) / sum((x + _EPS) ** -1 for x in xs))


class XView2PixelScorer:
    """Accumulates official xView2 pixel counts over tiles.

    Usage::

        scorer = XView2PixelScorer()
        for tile in tiles:
            scorer.add(loc_pred, dmg_pred, loc_target, dmg_target)
        summary = scorer.summary()

    All inputs are uint8 arrays of identical shape. loc_* are binary building
    masks (0/1, anything >0 counts as building); dmg_* carry grades 0-4 where
    0 = background and 1-4 = no/minor/major/destroyed damage inside buildings.
    """

    def __init__(self):
        self.loc_tp = 0
        self.loc_fp = 0
        self.loc_fn = 0
        self.dmg_tp = np.zeros(5, dtype=np.int64)   # index 1-4 used
        self.dmg_fp = np.zeros(5, dtype=np.int64)
        self.dmg_fn = np.zeros(5, dtype=np.int64)
        self.n_tiles = 0

    def add(self, loc_pred, dmg_pred, loc_target, dmg_target) -> None:
        """Add one tile (any spatial shape, pred and target must match)."""
        loc_pred = np.asarray(loc_pred)
        dmg_pred = np.asarray(dmg_pred)
        loc_target = np.asarray(loc_target)
        dmg_target = np.asarray(dmg_target)
        if not (loc_pred.shape == dmg_pred.shape == loc_target.shape == dmg_target.shape):
            raise ValueError(
                f"shape mismatch: pred {loc_pred.shape}/{dmg_pred.shape} vs "
                f"target {loc_target.shape}/{dmg_target.shape}"
            )
        self.n_tiles += 1

        lp_b = loc_pred > 0
        lt_b = loc_target > 0
        self.loc_tp += int(np.logical_and(lp_b, lt_b).sum())
        self.loc_fp += int(np.logical_and(lp_b, ~lt_b).sum())
        self.loc_fn += int(np.logical_and(~lp_b, lt_b).sum())

        dp = np.where(lp_b, dmg_pred, 0)          # gate damage by predicted buildings
        dt = dmg_target
        inside = dt > 0                            # score only target-building pixels
        dp_in = dp[inside]
        dt_in = dt[inside]
        for c in range(1, 5):
            p_c = dp_in == c
            g_c = dt_in == c
            self.dmg_tp[c] += int(np.logical_and(p_c, g_c).sum())
            self.dmg_fn[c] += int(np.logical_and(~p_c, g_c).sum())
            self.dmg_fp[c] += int(np.logical_and(p_c, ~g_c).sum())

    def summary(self) -> dict:
        localization_f1 = f1_from_counts(self.loc_tp, self.loc_fp, self.loc_fn)
        per_class = {
            c: f1_from_counts(int(self.dmg_tp[c]), int(self.dmg_fp[c]), int(self.dmg_fn[c]))
            for c in range(1, 5)
        }
        damage_f1 = harmonic_mean_f1(per_class.values())
        return {
            "score": 0.3 * localization_f1 + 0.7 * damage_f1,
            "localization_f1": localization_f1,
            "damage_f1": damage_f1,
            "damage_f1_no_damage": per_class[1],
            "damage_f1_minor_damage": per_class[2],
            "damage_f1_major_damage": per_class[3],
            "damage_f1_destroyed": per_class[4],
            "n_tiles": self.n_tiles,
        }

    def __repr__(self) -> str:
        s = self.summary()
        per_class = [s["damage_f1_no_damage"], s["damage_f1_minor_damage"],
                     s["damage_f1_major_damage"], s["damage_f1_destroyed"]]
        return (
            f"XView2PixelScorer(n={s['n_tiles']}) score={s['score']:.4f} "
            f"loc_f1={s['localization_f1']:.4f} dmg_f1={s['damage_f1']:.4f} "
            f"per-class={per_class}"
        )


print("official scorer ready:", XView2PixelScorer)
mark("scorer:done")

In [ ]:
import torch as _torch
import torch.nn as _nn
import torch.nn.functional as _F
import segmentation_models_pytorch as _smp


class SiameseUNet(_nn.Module):
    """Shared-encoder siamese U-Net with fused pre/post features (fixed decoder)."""

    def __init__(self, encoder_name="resnet34", encoder_weights="imagenet",
                 fusion_mode="concat_absdiff", classes=1):
        super().__init__()
        self.fusion_mode = fusion_mode
        self.classes = classes

        base_unet = _smp.Unet(encoder_name=encoder_name, encoder_weights=encoder_weights,
                              in_channels=3, classes=classes)
        self.encoder = base_unet.encoder
        encoder_channels = self._get_encoder_channels()
        bottleneck_ch = encoder_channels[-1]

        if fusion_mode == "concat":
            fusion_mult = 2
        elif fusion_mode == "absdiff":
            fusion_mult = 1
        else:
            fusion_mult = 3   # concat_absdiff

        skip_channels = list(reversed(encoder_channels[:-1]))
        decoder_channels = [256, 128, 64, 32]

        self.skip_convs = _nn.ModuleList()
        for enc_ch in skip_channels:
            self.skip_convs.append(_nn.Sequential(
                _nn.Conv2d(enc_ch * fusion_mult, enc_ch, 1, bias=False),
                _nn.BatchNorm2d(enc_ch),
                _nn.ReLU(inplace=True),
            ))

        self.decoder = _nn.ModuleList()
        in_ch = bottleneck_ch * fusion_mult + skip_channels[0]
        for i, dec_ch in enumerate(decoder_channels):
            self.decoder.append(_nn.Sequential(
                _nn.Conv2d(in_ch, dec_ch, 3, padding=1, bias=False),
                _nn.BatchNorm2d(dec_ch),
                _nn.ReLU(inplace=True),
                _nn.Conv2d(dec_ch, dec_ch, 3, padding=1, bias=False),
                _nn.BatchNorm2d(dec_ch),
                _nn.ReLU(inplace=True),
            ))
            if i + 1 < len(decoder_channels):
                in_ch = dec_ch + skip_channels[i + 1]

        self.segmentation_head = _nn.Conv2d(decoder_channels[-1], classes, 1)

    def _get_encoder_channels(self):
        channels = []
        dummy = _torch.randn(1, 3, 64, 64)
        with _torch.no_grad():
            features = self.encoder(dummy)
        return [f.shape[1] for f in features]

    def _fuse(self, a, b):
        if self.fusion_mode == "concat":
            return _torch.cat([a, b], dim=1)
        if self.fusion_mode == "absdiff":
            return _torch.abs(b - a)
        return _torch.cat([a, b, _torch.abs(b - a)], dim=1)

    def forward(self, pre_image, post_image):
        features_pre = self.encoder(pre_image)
        features_post = self.encoder(post_image)
        x = self._fuse(features_pre[-1], features_post[-1])
        for i, (dec_block, skip_conv) in enumerate(zip(self.decoder, self.skip_convs)):
            x = _F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
            skip = skip_conv(self._fuse(features_pre[-(i + 2)], features_post[-(i + 2)]))
            if x.shape[2:] != skip.shape[2:]:
                x = _F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=False)
            x = dec_block(_torch.cat([x, skip], dim=1))
        if x.shape[2:] != pre_image.shape[2:]:
            x = _F.interpolate(x, size=pre_image.shape[2:], mode="bilinear", align_corners=False)
        return self.segmentation_head(x)


print("SiameseUNet (fixed decoder) defined")

In [ ]:
# ---- joint damage dataset (6ch siamese input, combined building+grade mask) ----
import pandas as pd
import torch
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2
from rescuevision.constants import IMAGENET_MEAN, IMAGENET_STD


class JointDamageDataset(Dataset):
    """pre+post -> 6ch image; ONE combined mask carries building + grade.
    values: 0 bg | 1 no-dmg | 2 minor | 3 major | 4 destroyed | 6 building w/o grade"""

    def __init__(self, root, split_csv, split="train", transform=None):
        self.root = Path(root)
        df = pd.read_csv(split_csv)
        ids = df[df["split"] == split]["tile_id"].tolist()

        def _ok(t):
            try:
                _find_tile(self.root / "images/pre", t)
                _find_tile(self.root / "masks/damage_multiclass_dil", t)
                return True
            except FileNotFoundError:
                return False

        self.tile_ids = [t for t in ids if _ok(t)]
        self.transform = transform
        print(f"JointDamageDataset split={split}: {len(self.tile_ids)} tiles")

    def __len__(self):
        return len(self.tile_ids)

    def __getitem__(self, idx):
        tid = self.tile_ids[idx]
        pre = _load_arr(_find_tile(self.root / "images/pre", tid))
        post = _load_arr(_find_tile(self.root / "images/post", tid))
        dmg = _load_arr(_find_tile(self.root / "masks/damage_multiclass_dil", tid)).astype(np.uint8)
        loc = _load_arr(self.root / "masks/building_binary" / f"{tid}.png").astype(np.uint8)
        comb = np.where((loc > 0) & (dmg == 0), 6, dmg).astype(np.uint8)
        pair = np.concatenate([pre, post], axis=2)
        if self.transform:
            aug = self.transform(image=pair, mask=comb)
            pair_t, comb_t = aug["image"], aug["mask"]
        else:
            pair_t = torch.from_numpy(pair.transpose(2, 0, 1))
            comb_t = torch.from_numpy(comb)
        comb_t = comb_t.long() if torch.is_tensor(comb_t) else torch.from_numpy(np.asarray(comb_t)).long()
        loc_t = (comb_t > 0).float()
        dmg_t = comb_t.clone() - 1
        dmg_t[(comb_t < 1) | (comb_t > 4)] = 255
        return {"pre": pair_t[:3].float(), "post": pair_t[3:].float(),
                "loc": loc_t, "dmg": dmg_t, "tile_id": tid}


def joint_pair_transforms(phase="train", size=608):
    mean, std = IMAGENET_MEAN * 2, IMAGENET_STD * 2
    tf = [A.Resize(size, size)]
    if phase == "train":
        tf += [A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
               A.Affine(scale=(0.85, 1.15), p=0.4),
               A.RandomBrightnessContrast(p=0.3),
               A.UnsharpMask(p=0.3)]
    tf += [A.Normalize(mean=mean, std=std), ToTensorV2()]
    return A.Compose(tf, is_check_shapes=False)


print("dataset kit ready")
mark("dataset:done")

In [ ]:
import time
import pandas as pd
mark("tar:wait")
TAR = None
if (Path("/tmp/processed_xbd") / "metadata" / "splits_event_aware.csv").exists():
    print("processed tree already at /tmp/processed_xbd (smoke/debug path)")
else:
    for _attempt in range(40):
        for _p in Path("/kaggle/input").rglob("processed_xbd*.tar"):
            TAR = _p
            break
        if TAR is not None:
            break
        _ents = [p.name for p in Path("/kaggle/input").iterdir()] if Path("/kaggle/input").exists() else []
        print(f"[{_attempt+1}/40] waiting for tar mount, /kaggle/input: {_ents}", flush=True)
        time.sleep(30)
    assert TAR is not None, "processed_xbd_native.tar never mounted - kernel-source attach failed"
    _gib = TAR.stat().st_size / 1e9
    print("tar:", TAR, f"{_gib:.2f} GB")
    mark("tar:mounted", extra=f"{_gib:.2f}GB")
    subprocess.run(["tar", "-xf", str(TAR), "-C", "/tmp"], check=True)
PROCESSED = Path("/tmp/processed_xbd")
assert (PROCESSED / "metadata" / "splits_event_aware.csv").exists(), "extracted tree missing splits csv"
_splits = pd.read_csv(PROCESSED / "metadata" / "splits_event_aware.csv")
print("tiles per split:", _splits["split"].value_counts().to_dict())
_tiles_meta = pd.read_csv(PROCESSED / "metadata" / "tiles.csv")
print("shift columns:", [c for c in _tiles_meta.columns if "shift" in c])
mark("tar:extracted", extra=str(_splits["split"].value_counts().to_dict()))

In [ ]:
# ---- locate the joint checkpoint: kernel-source mount first, /tmp fallback (smoke)
CKPT = None
for base in (Path("/kaggle/input/notebooks"), Path("/tmp")):
    if base.exists():
        for cand in base.glob("**/joint_damage_siamese_full/checkpoints/best.pt"):
            CKPT = cand
            break
    if CKPT is not None:
        break
assert CKPT is not None, "best.pt not found (attach rv2-joint-damage-full kernel source)"
print("checkpoint:", CKPT, f"{CKPT.stat().st_size/1e6:.1f}MB")
mark("ckpt:found", extra=str(CKPT))

In [ ]:
# ---- model kit: build SiameseUNet + TTA inference + prob caching ----
import time as _time
import numpy as np
import torch
from torch.utils.data import DataLoader

SIZE = 608

def build_model(encoder="resnet34", weights="none", se_gates=False):
    weights = None if weights in (None, "none", "") else weights
    cls = SiameseUNetSE if se_gates else SiameseUNet
    m = cls(encoder_name=encoder, encoder_weights=weights,
            fusion_mode="concat_absdiff", classes=5)
    return m.to(DEVICE).eval()


@torch.no_grad()
def tta_probs(model, pre, post):
    """4-view TTA -> (loc_prob, dmg_prob[4]) from averaged logits."""
    outs = []
    for k in range(4):
        a, b = pre, post
        if k == 1: a, b = torch.flip(pre, [3]), torch.flip(post, [3])
        elif k == 2: a, b = torch.flip(pre, [2]), torch.flip(post, [2])
        elif k == 3: a, b = torch.flip(pre, [2, 3]), torch.flip(post, [2, 3])
        with torch.amp.autocast(DEVICE.type, enabled=(DEVICE.type == "cuda")):
            o = model(a, b)
        if k == 1: o = torch.flip(o, [3])
        elif k == 2: o = torch.flip(o, [2])
        elif k == 3: o = torch.flip(o, [2, 3])
        outs.append(o.float())
    o = torch.stack(outs, 0).mean(0)
    return torch.sigmoid(o[:, 0]), torch.softmax(o[:, 1:], 1)


@torch.no_grad()
def cache_probs(model, ds, split, limit=None, tag="probs"):
    """TTA inference over ds.tile_ids, per-tile probs cached as float16 npz in /tmp."""
    if limit:
        ds.tile_ids = ds.tile_ids[:limit]
    loader = DataLoader(ds, batch_size=4, shuffle=False, num_workers=NW)
    ids = []
    t0 = _time.time()
    n_batches = 0
    for b in loader:
        loc_p, dmg_p = tta_probs(model, b["pre"].to(DEVICE), b["post"].to(DEVICE))
        loc_np = loc_p.cpu().numpy().astype(np.float16)
        dmg_np = dmg_p.cpu().numpy().astype(np.float16)
        for i in range(loc_np.shape[0]):
            tid = b["tile_id"][i]
            np.savez_compressed(Path("/tmp") / f"rescore_{split}_{tid}.npz",
                                loc=loc_np[i], dmg=dmg_np[i])
            ids.append(tid)
        n_batches += 1
        if n_batches % 5 == 0:
            el = _time.time() - t0
            mark(f"cache:{tag}", extra=f"{len(ids)} tiles, {el/60:.1f}min, "
                                        f"~{el/max(len(ids),1):.1f}s/tile")
    mark(f"cache:{tag}-done", extra=f"{len(ids)} tiles in {(_time.time()-t0)/60:.1f}min")
    return ids


def cached_ids(tag, split):
    prefix = f"{tag}_{split}_"
    return sorted(p.name.removeprefix(prefix).removesuffix(".npz")
                  for p in Path("/tmp").glob(prefix + "*.npz"))


mark("modelkit:done")

In [ ]:
# ---- run TTA inference over the full event-disjoint TEST split (CPU) ----
from rescuevision.utils.seed import set_seed

split_csv = PROCESSED / "metadata" / "splits_event_aware.csv"
model = build_model("resnet34", "none")
ckpt = torch.load(CKPT, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
print("loaded ckpt epoch", ckpt.get("epoch"), "val", ckpt.get("val", {}).get("loc_iou"))
mark("ckpt:loaded", extra=f"epoch={ckpt.get('epoch')}")

set_seed(42)
test_ds = JointDamageDataset(PROCESSED, split_csv, "test", joint_pair_transforms("val", SIZE))
ids = cache_probs(model, test_ds, "test", limit=None, tag="rescore")
print("cached", len(ids), "test tiles")
del model
mark("rescore:inference-done", extra=str(len(ids)))

In [ ]:
# ---- official scoring over cached probs: threshold sweep + winner rule +
# ---- inference-time minor dilation, against original AND dilated targets.
import json
from functools import lru_cache
import numpy as np
from PIL import Image
from scipy.ndimage import binary_dilation
import cv2

SCORING_SPLIT = "test"
THR_LIST = [round(t, 2) for t in np.arange(0.3, 0.85 + 1e-9, 0.05)]
DILATE_ITERS = 2
TAG = "rescore"


@lru_cache(maxsize=None)
def load_gt(tid):
    bld = np.array(Image.open(PROCESSED / "masks/building_binary" / f"{tid}.png"))
    dmg = np.array(Image.open(PROCESSED / "masks/damage_multiclass" / f"{tid}.png"))
    dmgd = np.array(Image.open(PROCESSED / "masks/damage_multiclass_dil" / f"{tid}.png"))
    return bld, dmg, dmgd


def to_native_2d(arr, shape):
    if arr.shape == shape:
        return arr
    return cv2.resize(arr.astype(np.float32), (shape[1], shape[0]),
                      interpolation=cv2.INTER_LINEAR)


def to_native_3d(arr, shape):
    """(C,h,w) channels-first -> bilinear resample each channel to shape."""
    if arr.shape[1:] == shape:
        return arr
    return np.stack([cv2.resize(arr[i].astype(np.float32), (shape[1], shape[0]),
                                interpolation=cv2.INTER_LINEAR)
                     for i in range(arr.shape[0])])


@lru_cache(maxsize=None)
def native_preds(tid, prefix=None):
    """Per-tile native-resolution inputs, computed once and shared by all
    scoring variants: (bld, dmg, dmgd, loc_prob, grade_1to4).
    prefix defaults to the current TAG_SPLIT (pass e.g. 'eval_test' to read
    another split's cached probs)."""
    prefix = prefix or f"{TAG}_{SCORING_SPLIT}"
    z = np.load(Path("/tmp") / f"{prefix}_{tid}.npz")
    bld, dmg, dmgd = load_gt(tid)
    shape = bld.shape
    loc = to_native_2d(z["loc"].astype(np.float32), shape)
    dmg_p = to_native_3d(z["dmg"].astype(np.float32), shape)  # (4,H,W)
    grade = np.argmax(dmg_p, axis=0) + 1
    return bld, dmg, dmgd, loc, grade


def winner_building_mask(loc_prob, grade):
    """xView2-winner conditional rule, EXACT (create_submission.py): building if
    loc>0.38 OR (loc>0.13 & grade in {2 minor, 3 major}) OR (loc>0.14 & grade>=2)."""
    return ((loc_prob > 0.38) |
            ((loc_prob > 0.13) & (grade >= 2) & (grade <= 3)) |
            ((loc_prob > 0.14) & (grade >= 2)))


def dilate_minor_pred(grade, iters=DILATE_ITERS):
    """winner-exact: 5x5 dilation of the MINOR(2) mask; only no-damage(1)
    pixels inside the dilated mask flip to minor (bg and 3/4 untouched)."""
    if iters <= 0:
        return grade, np.zeros_like(grade, dtype=bool)
    st = np.ones((5, 5), dtype=bool)
    grow = binary_dilation(grade == 2, structure=st) & (grade == 1)
    out = grade.copy()
    out[grow] = 2
    return out, grow


def score_variant(ids, thr, rule, dilate, target_key):
    """rule: 'plain' (thr on loc) or 'winner'. target_key: 'dmg' (original
    annotations = official) or 'dmgd' (dilated training labels, reference)."""
    sc = XView2PixelScorer()
    for tid in ids:
        bld, dmg, dmgd, loc, grade = native_preds(tid)
        build = winner_building_mask(loc, grade) if rule == "winner" else (loc > thr)
        grade = np.where(build, grade, 0)
        if dilate:
            grade, grow = dilate_minor_pred(grade)
            build = build | grow
        tgt = dmg if target_key == "dmg" else dmgd
        sc.add(build.astype(np.uint8), grade.astype(np.uint8),
               bld.astype(np.uint8), tgt.astype(np.uint8))
    return sc.summary()


ids = cached_ids(TAG, SCORING_SPLIT)
assert len(ids) > 0, "no cached probs found"
print(f"scoring {len(ids)} tiles of split '{SCORING_SPLIT}'")
_ = [native_preds(t) for t in ids]  # warm the per-tile cache once

results = {}
for target_key in ("dmg", "dmgd"):
    variants = [("plain", t) for t in THR_LIST] + [("winner", None)]
    for rule, thr in variants:
        for dilate in ((False, True) if DILATE_ITERS > 0 else (False,)):
            key = f"tgt={target_key}|rule={rule}|thr={thr}|dilate={int(dilate)}"
            r = score_variant(ids, thr, rule, dilate, target_key)
            results[key] = r
            print(f"{key:44s} score={r['score']:.4f} loc={r['localization_f1']:.4f} "
                  f"dmg={r['damage_f1']:.4f}")

best_key = max(results, key=lambda k: results[k]["score"])
official = results[best_key]
print("\nBEST RECIPE:", best_key)
print(json.dumps(official, indent=2))

OFFICIAL_RESULTS = {"best_recipe": best_key, "best": official,
                    "n_tiles": len(ids), "split": SCORING_SPLIT,
                    "all_variants": results}
(Path("/tmp") / f"{TAG}_official_{SCORING_SPLIT}.json").write_text(
    json.dumps(OFFICIAL_RESULTS, indent=1))
mark(f"official:{SCORING_SPLIT}-done",
     extra=f"best={best_key} score={official['score']:.4f}")

In [ ]:
# ---- publish results ----
import shutil
shutil.copyfile("/tmp/rescore_official_test.json", WORK / "rescore_official_test.json")
print("written:", WORK / "rescore_official_test.json")
mark("rescore:ALL-DONE")